# Telecom Customer Churn Prediction

This notebook demonstrates an end-to-end ML pipeline for predicting customer churn
in a telecommunications network using network quality indicators, customer behavior,
and service metrics.

## 1. Setup & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("notebook")
sns.set_style("whitegrid")
sns.set_palette("husl")

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
import sys
from pathlib import Path

# Add project source to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
RANDOM_STATE = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 2. Data Loading & Validation

In [ ]:
df = pd.read_parquet(DATA_DIR / "synthetic_data.parquet")
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
print("Column data types:")
print("=" * 40)
print(df.dtypes)

In [ ]:
df.describe().round(3)

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"count": missing, "pct": missing_pct})
print("Missing values:")
print(missing_df[missing_df["count"] > 0] if missing.sum() > 0 else "No missing values found.")

In [ ]:
# Check churn rate
churn_counts = df["is_churned"].value_counts()
churn_rate = df["is_churned"].mean() * 100
print(f"Churn distribution:")
print(churn_counts)
print(f"\nOverall churn rate: {churn_rate:.2f}%")
print(f"Class imbalance ratio: 1:{(1 - df['is_churned'].mean()) / df['is_churned'].mean():.1f}")

## 3. Exploratory Data Analysis

In [ ]:
# Churn distribution barplot
fig, ax = plt.subplots(figsize=(8, 5))
churn_labels = {0: "Not Churned", 1: "Churned"}
plot_df = df["is_churned"].map(churn_labels).value_counts()
sns.barplot(x=plot_df.index, y=plot_df.values, ax=ax)
ax.set_title("Customer Churn Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Churn Status")
ax.set_ylabel("Count")
for i, v in enumerate(plot_df.values):
    ax.text(i, v + len(df) * 0.01, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by churn status
features_to_plot = ["sinr", "qoe_mos", "tenure"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feature in zip(axes, features_to_plot):
    if feature in df.columns:
        for label, group in df.groupby("is_churned"):
            ax.hist(group[feature], bins=40, alpha=0.6,
                    label=f"{'Churned' if label == 1 else 'Not Churned'}")
        ax.set_title(f"{feature} by Churn Status", fontsize=12, fontweight="bold")
        ax.set_xlabel(feature)
        ax.set_ylabel("Frequency")
        ax.legend()
    else:
        ax.text(0.5, 0.5, f"{feature}\nnot found", ha="center", va="center",
                transform=ax.transAxes, fontsize=12)
        ax.set_title(f"{feature} (not available)")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with churn
if "is_churned" in corr_matrix.columns:
    churn_corr = corr_matrix["is_churned"].drop("is_churned").sort_values(key=abs, ascending=False)
    print("Top features correlated with churn:")
    print("=" * 40)
    for feat, val in churn_corr.head(10).items():
        direction = "(+)" if val > 0 else "(-)"
        print(f"  {feat:30s} {val:+.4f} {direction}")

## 4. Feature Engineering

In [ ]:
from churn_prediction.feature_engineer import FeatureEngineer

fe = FeatureEngineer()
df_features = fe.pipeline(df)

print(f"Shape before feature engineering: {df.shape}")
print(f"Shape after feature engineering:  {df_features.shape}")
print(f"New features added: {df_features.shape[1] - df.shape[1]}")

In [ ]:
# Display new feature columns
original_cols = set(df.columns)
new_cols = [c for c in df_features.columns if c not in original_cols]
print(f"New feature columns ({len(new_cols)}):")
print("=" * 40)
for col in new_cols:
    print(f"  - {col}")

In [ ]:
# Preview the engineered features
if new_cols:
    df_features[new_cols].describe().round(3)

## 5. Model Training

In [ ]:
from churn_prediction.model import XGBoostChurnClassifier

model = XGBoostChurnClassifier(random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = model.prepare_data(
    df_features, target="is_churned"
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.4f}")
print(f"Test churn rate:  {y_test.mean():.4f}")

In [ ]:
# Train the model
model.train(X_train, y_train)
print("Model training complete.")
print(f"Number of features: {len(model.feature_names)}")

In [ ]:
# Generate predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

print(f"Predictions generated for {len(y_pred):,} test samples.")
print(f"Predicted churn rate: {y_pred.mean():.4f}")
print(f"Mean predicted probability: {y_prob.mean():.4f}")

## 6. Evaluation & Metrics

In [ ]:
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve
)

auroc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Performance Metrics")
print("=" * 40)
print(f"  AUROC:     {auroc:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Not Churned", "Churned"]))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, linewidth=2, label=f"XGBoost (AUROC = {auroc:.4f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve - Churn Prediction", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Not Churned", "Churned"],
            yticklabels=["Not Churned", "Churned"])
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual", fontsize=12)
ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Interpretation (SHAP)

In [ ]:
import shap

explainer = shap.TreeExplainer(model.model)
shap_values = explainer.shap_values(X_test)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Number of features explained: {shap_values.shape[1]}")

In [ ]:
# SHAP summary plot
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type="dot",
                  max_display=15, show=False)
plt.title("SHAP Feature Importance - Churn Prediction", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP waterfall plot for a single prediction
sample_idx = 0
expected_value = explainer.expected_value
if isinstance(expected_value, np.ndarray):
    expected_value = expected_value[0]

explanation = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=expected_value,
    data=X_test.iloc[sample_idx],
    feature_names=X_test.columns.tolist()
)

print(f"Prediction for sample {sample_idx}:")
print(f"  Actual label:       {y_test.iloc[sample_idx]}")
print(f"  Predicted prob:     {y_prob[sample_idx]:.4f}")
print()

fig, ax = plt.subplots(figsize=(12, 6))
shap.plots.waterfall(explanation, max_display=12, show=False)
plt.title("SHAP Waterfall - Single Customer Prediction", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Business Insights & Conclusions

### Key Findings

1. **Churn Drivers**: Network quality metrics (SINR, QoE MOS) are among the strongest
   predictors of customer churn, confirming that service experience directly impacts retention.

2. **Tenure Effect**: Customers with shorter tenure are significantly more likely to churn,
   suggesting that the first months of service are critical for retention.

3. **Model Performance**: The XGBoost classifier achieves strong AUROC, indicating reliable
   discrimination between churners and non-churners.

4. **Feature Engineering Impact**: Engineered features derived from raw network KPIs
   capture interaction effects that improve predictive accuracy.

### Business Recommendations

- **Proactive Retention**: Deploy the model to score customers weekly and trigger
  retention campaigns for high-risk individuals before they decide to leave.

- **Network Quality Investment**: Prioritize infrastructure improvements in regions
  where SINR and throughput metrics are consistently low, as these areas likely
  generate the most churn.

- **Onboarding Programs**: Implement enhanced onboarding and support for new
  subscribers during their first 3-6 months to reduce early-tenure churn.

- **Personalized Offers**: Use SHAP explanations per customer to craft personalized
  retention offers addressing their specific pain points (e.g., data plan upgrades
  for heavy users with congestion issues).

- **Monitoring Dashboard**: Create a real-time churn risk dashboard integrating
  the model predictions with CRM systems for customer-facing teams.

In [ ]:
# Summary statistics
print("Churn Prediction Model Summary")
print("=" * 50)
print(f"Dataset size:          {len(df):,} customers")
print(f"Features used:         {X_train.shape[1]}")
print(f"Churn rate:            {churn_rate:.2f}%")
print(f"AUROC:                 {auroc:.4f}")
print(f"F1 Score:              {f1:.4f}")
print(f"Precision:             {precision:.4f}")
print(f"Recall:                {recall:.4f}")
print("=" * 50)
print("Model is ready for deployment evaluation.")